<a href="https://colab.research.google.com/github/YaminisriVellanki/24B11DS221_DriveReady_OPC_2026/blob/main/task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# DAY 3 – YOUR FIRST AGENT: WHAT AN AGENT IS, AND LANGCHAIN

## Introduction

Today, I am learning about AI Agents and the LangChain framework.

In Day 3, I will understand the difference between a chatbot and an AI agent, learn the four parts of an agent, understand the agent loop, and explore how LangChain works.

## Concepts Covered

1. Chatbot vs AI Agent
2. Four Parts of an Agent
3. Agent Loop – Think, Act, Observe, Repeat
4. Three Levels of AI
5. Agent or Not?
6. Introduction to LangChain
7. LangChain Model, Prompt Template and Chain
8. Testing an AI Assistant using CSV

## Practical Implementation

- Connect the Groq API with LangChain.
- Build an AI assistant using LangChain.
- Implement conversation history.
- Create customer test cases using CSV.
- Customize the assistant for my own business, DemandPulse AI.
- Explore the LangChain agent.

## Tools Used

- Google Colab
- Python
- Groq API
- LangChain
- CSV



## Step 1 – Install Libraries and Connect Groq API

In this step, I am installing the required LangChain libraries and connecting the Groq API.

LangChain helps us connect language models with prompts, memory, tools and other components.

I am using the Groq API to access the language model for building my AI assistant.


In [2]:
!pip install -q langchain langchain-groq

In [7]:
import os
from google.colab import userdata

# Get API key from Google Colab Secrets
os.environ["GROQ_API_KEY"] = userdata.get("Agentic_ai_api_key").strip()

print("Groq API key configured successfully")

Groq API key configured successfully



## Step 2 – Initialize the Chat Model

In this step, I am importing ChatGroq and initializing the language model.

I am using the openai/gpt-oss-20b model with a temperature of 0.3 to generate responses.


In [8]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.3
)

print("LangChain model initialized successfully")

LangChain model initialized successfully



## Task 1 – Create the AI Assistant Prompt

In this step, I am importing the required LangChain components and defining the SYSTEM prompt for Sweet Crumbs Home Bakery.

The SYSTEM prompt contains:
- ROLE: Defines the assistant's role.
- FACTS: Provides bakery information.
- RULES: Defines what the assistant should and should not do.
- FORMAT: Defines how the assistant should respond.

This prompt guides the AI assistant to provide accurate and friendly customer support.


In [9]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

# Define the system prompt
SYSTEM = """ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.

FACTS:  Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80 each.
        Orders need 24 hours' notice. All cakes have eggless versions.

RULES:  Never promise a delivery time or place. If it isn't in FACTS,
        say "I'll check with the owner." Work out totals carefully.

FORMAT: Under 50 words, friendly, English or Telugu matching the customer.
"""

print("System prompt created successfully")

System prompt created successfully



## Step 2 – Create Prompt Template and Chain

In this step, I am creating a ChatPromptTemplate to organize the system message, conversation history and customer question.

MessagesPlaceholder is used to insert previous conversation messages.

The chain connects three components:
1. Prompt template
2. Language model
3. String output parser

The pipe operator (|) connects these components into one workflow.


In [10]:
# Create the prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

# Create the LangChain chain
chain = prompt | llm | StrOutputParser()

print("Prompt template and chain created successfully")

Prompt template and chain created successfully



## Step 3 – Implement Conversation Memory

In this step, I am creating a history list to store previous customer messages and AI responses.

I am defining an ask() function that:
- Accepts a customer question.
- Sends the question along with previous conversation history to the chain.
- Stores the customer question and AI response in history.
- Returns the AI response.

This allows the assistant to remember information from the current conversation.


In [11]:
# Initialize conversation history
history = []

# Define the assistant function
def ask(question: str) -> str:
    answer = chain.invoke({
        "history": history,
        "question": question
    })

    # Store conversation messages
    history.extend([
        ("human", question),
        ("ai", answer)
    ])

    return answer


## Step 4 – Test the AI Assistant

In this step, I am testing the chatbot with two customer questions.

First, I introduce myself as Ravi and ask about eggless cakes.

Next, I ask the assistant to remember my name.

This helps verify whether conversation history is working correctly.


In [12]:
print(ask("Hi, I'm Ravi. Do you make eggless cakes?"))

print(ask("What's my name?"))

Yes, all our cakes are eggless. We offer plum cake (₹500), chocolate cake (₹650), and cupcakes (₹80 each). Orders need 24 hours notice.
Your name is Ravi.



## Step 5 – Display Conversation History

In this step, I am displaying the history list to check whether customer messages and AI responses have been stored.

This helps verify the memory implementation.


In [13]:
history

[('human', "Hi, I'm Ravi. Do you make eggless cakes?"),
 ('ai',
  'Yes, all our cakes are eggless. We offer plum cake (₹500), chocolate cake (₹650), and cupcakes (₹80 each). Orders need 24\u202fhours notice.'),
 ('human', "What's my name?"),
 ('ai', 'Your name is Ravi.')]


## Task 2 – Test the AI Assistant Using CSV

In this task, I am using the test.csv file provided by the instructor.

The file contains customer questions and expected keywords.

I am testing the assistant by comparing its responses with the expected keywords to calculate the test pass rate.


In [16]:
import csv

def run_tests(path: str = "/content/test.csv") -> None:
    rows = list(csv.DictReader(open(path)))
    passed = 0

    for row in rows:
        answer = chain.invoke({
            "history": [],
            "question": row["question"]
        })

        ok = row["must_include"].lower() in answer.lower()
        passed += ok

        print(
            "PASS" if ok else "FAIL",
            "|",
            row["question"],
            "→",
            answer[:70]
        )

    print(f"\n{passed}/{len(rows)} passed ({passed / len(rows):.0%})")

In [17]:
run_tests()

PASS | How much is a plum cake? → A plum cake costs Rs 500.
FAIL | Do you make eggless cakes? → Yes, we offer egg‑less versions of all our cakes—plum, chocolate, and 
FAIL | Can I get a cake today evening? → I’m sorry, we need 24‑hour notice for orders. I’ll check with the owne
PASS | Do you deliver to Kakinada? → I’ll check with the owner and get back to you.
PASS | What is the price of 6 cupcakes? → Each cupcake is Rs 80, so 6 cupcakes cost Rs 480.

3/5 passed (60%)



## Step 3 – Create an AI Agent Using LangChain

In this step, I am creating an AI agent using LangChain's create_agent function.

I am providing the language model and SYSTEM prompt to the agent.

The agent receives a customer question and generates a response based on the instructions.

In this example, no external tools are provided to the agent.


In [18]:
from langchain.agents import create_agent

# Create the LangChain agent
agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=SYSTEM
)

print("LangChain agent created successfully")

LangChain agent created successfully



## Step 4 – Test the LangChain Agent

In this step, I am testing the agent by asking a customer question about eggless cakes.

I am checking whether the agent follows the SYSTEM prompt and provides the correct bakery information.


In [19]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Do you make eggless cakes?"
        }
    ]
})

print(result["messages"][-1].content)

Yes, we offer egg‑free versions of all our cakes – plum, chocolate, and cupcakes. Let me know what you'd like!



## Task 1 – Customize the AI Assistant for DemandPulse AI

In this task, I am customizing the AI assistant for my own business, DemandPulse AI.

I am replacing the bakery SYSTEM prompt with my business details, including services, prices, rules and response format.

This helps me understand how an AI assistant can be adapted to different business requirements.


In [20]:
# Task 1: DemandPulse AI SYSTEM Prompt

SYSTEM = """
ROLE:
You are a friendly AI assistant for DemandPulse AI.

FACTS:
DemandPulse AI provides the following services:

1. Sales Forecasting - Rs 500
2. Demand Analysis - Rs 300
3. Inventory Planning - Rs 400
4. Business Report - Rs 200

RULES:
- Provide information only about the listed services.
- Do not invent service prices.
- If information is unavailable, say "I'll check with the team."
- Explain services in simple English.
- Calculate totals carefully.

FORMAT:
Keep answers under 50 words.
Be friendly, clear and professional.
"""

print("DemandPulse AI SYSTEM prompt created successfully")

DemandPulse AI SYSTEM prompt created successfully



## Rebuild the Prompt Template and Chain

In this step, I am updating the prompt template with the DemandPulse AI SYSTEM prompt.

I am connecting the updated prompt, language model and output parser to create a new chain for my business assistant.


In [21]:
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

print("DemandPulse AI chain created successfully")

DemandPulse AI chain created successfully



##– Test the DemandPulse AI Assistant

In this step, I am testing my customized assistant by asking about its services and their prices.

This verifies whether the assistant follows my business SYSTEM prompt.


In [22]:
history = []

print(ask("What services does DemandPulse AI provide?"))

print(ask("What is the price of Sales Forecasting?"))

DemandPulse AI offers four services:  

1. **Sales Forecasting** – Rs 500  
2. **Demand Analysis** – Rs 300  
3. **Inventory Planning** – Rs 400  
4. **Business Report** – Rs 200  

Let me know if you’d like more details!
Sales Forecasting costs **Rs 500**. Let me know if you’d like to add it to a bundle or need more details!


## Task 2: Create Customer Test Questions

In this task, I am creating 10 sample customer questions for DemandPulse AI.
Each question has an expected keyword to check whether the AI assistant
provides the required information.

In [23]:
import csv

test_cases = [
    ["What services do you provide?", "Sales Forecasting"],
    ["What is the price of Sales Forecasting?", "500"],
    ["How much does Demand Analysis cost?", "300"],
    ["What is the price of Inventory Planning?", "400"],
    ["How much is a Business Report?", "200"],
    ["Can you help me forecast sales?", "Sales Forecasting"],
    ["Do you provide inventory planning?", "Inventory Planning"],
    ["Can you prepare a business report?", "Business Report"],
    ["What is the price of all four services?", "1400"],
    ["Can you provide financial auditing?", "check with the team"]
]

with open("demandpulse_tests.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["question", "must_include"])
    writer.writerows(test_cases)

print("Created 10 customer test questions successfully.")

Created 10 customer test questions successfully.


## Task 3: Testing the DemandPulse AI Assistant

In this task, I am testing my DemandPulse AI assistant using 10 customer questions stored in a CSV file.

I will:
- Read questions from the CSV file.
- Send each question to the LangChain model.
- Check whether the response contains the expected keyword.
- Display PASS or FAIL for each test.
- Calculate the overall test pass rate.

In [25]:
import csv

def run_tests(path="/content/demandpulse_tests.csv"):
    with open(path, "r") as file:
        rows = list(csv.DictReader(file))

    passed = 0

    for row in rows:
        answer = chain.invoke({
            "history": [],
            "question": row["question"]
        })

        expected = row["must_include"]
        ok = expected.lower() in answer.lower()

        if ok:
            passed += 1

        print("PASS" if ok else "FAIL", "|", row["question"])
        print("Expected:", expected)
        print("Response:", answer)
        print("-" * 50)

    print(f"Final Result: {passed}/{len(rows)} tests passed")
    print(f"Pass Rate: {passed / len(rows) * 100:.1f}%")

In [26]:
run_tests()

PASS | What services do you provide?
Expected: Sales Forecasting
Response: We offer four services:  
- Sales Forecasting – Rs 500  
- Demand Analysis – Rs 300  
- Inventory Planning – Rs 400  
- Business Report – Rs 200  

Let me know if you’d like more details!
--------------------------------------------------
PASS | What is the price of Sales Forecasting?
Expected: 500
Response: Sales Forecasting costs Rs 500. Let me know if you’d like details on any other service!
--------------------------------------------------
PASS | How much does Demand Analysis cost?
Expected: 300
Response: Demand Analysis costs Rs 300. Let me know if you’d like more details or a quote!
--------------------------------------------------
PASS | What is the price of Inventory Planning?
Expected: 400
Response: Inventory Planning costs Rs 400.
--------------------------------------------------
PASS | How much is a Business Report?
Expected: 200
Response: A Business Report costs Rs 200.
---------------------------

## Task 4: Improving the SYSTEM Prompt

In this task, I am improving the SYSTEM prompt of my DemandPulse AI assistant.

I will:
- Make the business facts clearer.
- Add specific rules for answering customer questions.
- Prevent the assistant from inventing services or prices.
- Run the same 10 CSV test cases again.
- Compare the new pass rate with the previous result.

In [27]:
SYSTEM = """
ROLE:
You are a professional and friendly customer support assistant
for DemandPulse AI.

FACTS:
1. Sales Forecasting costs Rs 500.
2. Demand Analysis costs Rs 300.
3. Inventory Planning costs Rs 400.
4. Business Report costs Rs 200.

RULES:
- Answer only using the business facts provided above.
- Never invent services or prices.
- If a customer asks about an unavailable service,
  reply: "I'll check with the team."
- If asked for the total price of all four services,
  calculate it as Rs 1400.
- Explain services in simple English.
- Keep answers accurate and relevant.

FORMAT:
- Keep responses under 50 words.
- Use a friendly and professional tone.
"""

In [28]:
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

print("DemandPulse AI prompt improved successfully.")

DemandPulse AI prompt improved successfully.


In [29]:
run_tests()

PASS | What services do you provide?
Expected: Sales Forecasting
Response: We offer four key services:

1. **Sales Forecasting** – predicts future sales.  
2. **Demand Analysis** – studies customer demand patterns.  
3. **Inventory Planning** – helps manage stock levels.  
4. **Business Report** – summarizes business performance.
--------------------------------------------------
PASS | What is the price of Sales Forecasting?
Expected: 500
Response: Sales Forecasting costs Rs 500.
--------------------------------------------------
PASS | How much does Demand Analysis cost?
Expected: 300
Response: Demand Analysis costs Rs 300.
--------------------------------------------------
PASS | What is the price of Inventory Planning?
Expected: 400
Response: Inventory Planning costs Rs 400.
--------------------------------------------------
PASS | How much is a Business Report?
Expected: 200
Response: A Business Report costs Rs 200. Let me know if you’d like to add any other services!
------------

## Task 5: Temperature Experiment

In this task, I am changing the temperature of my DemandPulse AI model from 0.3 to 1.0.

I will:
- Understand how temperature affects AI responses.
- Reinitialize the model with temperature 1.0.
- Rebuild the LangChain chain.
- Run the same 10 CSV test cases twice.
- Compare the results between both runs.

In [30]:
# Initialize the model with higher temperature
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=1.0
)

print("Model initialized with temperature 1.0")

Model initialized with temperature 1.0


In [31]:
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

print("LangChain updated successfully.")

LangChain updated successfully.


In [32]:
print("FIRST TEST RUN")
run_tests("/content/demandpulse_tests.csv")

FIRST TEST RUN
PASS | What services do you provide?
Expected: Sales Forecasting
Response: We offer four services:

- **Sales Forecasting** (predict future sales, Rs 500)
- **Demand Analysis** (study customer demand, Rs 300)
- **Inventory Planning** (manage stock levels, Rs 400)
- **Business Report** (summary insights, Rs 200)

All help you optimize your business decisions.
--------------------------------------------------
PASS | What is the price of Sales Forecasting?
Expected: 500
Response: Sales Forecasting costs Rs 500.
--------------------------------------------------
PASS | How much does Demand Analysis cost?
Expected: 300
Response: Demand Analysis costs Rs 300. It helps you see what customers want, so you can plan better. Let me know if you’d like more details or a quote!
--------------------------------------------------
PASS | What is the price of Inventory Planning?
Expected: 400
Response: Inventory Planning costs Rs 400.
--------------------------------------------------
PA

In [33]:
print("SECOND TEST RUN")
run_tests("/content/demandpulse_tests.csv")

SECOND TEST RUN
PASS | What services do you provide?
Expected: Sales Forecasting
Response: We offer four main services:  

1. **Sales Forecasting** – Rs 500  
2. **Demand Analysis** – Rs 300  
3. **Inventory Planning** – Rs 400  
4. **Business Report** – Rs 200  

Let me know which one you’re interested in!
--------------------------------------------------
PASS | What is the price of Sales Forecasting?
Expected: 500
Response: Sales Forecasting is priced at **Rs 500**.
--------------------------------------------------
PASS | How much does Demand Analysis cost?
Expected: 300
Response: Demand Analysis costs Rs 300. It helps you understand how customers might buy your products, so you can plan better.
--------------------------------------------------
PASS | What is the price of Inventory Planning?
Expected: 400
Response: Inventory Planning costs Rs 400.
--------------------------------------------------
PASS | How much is a Business Report?
Expected: 200
Response: The Business Report co

## Task 6: Agent or Not?

In this task, I am identifying three jobs in DemandPulse AI.

I will classify each job as:
1. Plain Code
2. One LLM Call
3. Agent

This helps me understand when to use normal Python code,
an LLM, or an AI agent.

In [34]:
jobs = [
    {
        "job": "Calculate the total price of services",
        "category": "Plain Code"
    },
    {
        "job": "Explain sales forecasting results to a customer",
        "category": "One LLM Call"
    },
    {
        "job": "Understand a customer request, check service information, "
               "generate a report, and save it",
        "category": "Agent"
    }
]

for item in jobs:
    print("Job:", item["job"])
    print("Category:", item["category"])
    print("-" * 40)

Job: Calculate the total price of services
Category: Plain Code
----------------------------------------
Job: Explain sales forecasting results to a customer
Category: One LLM Call
----------------------------------------
Job: Understand a customer request, check service information, generate a report, and save it
Category: Agent
----------------------------------------


## Task 7: Creating Python Functions for DemandPulse AI

In this task, I am creating two Python functions for my business,
DemandPulse AI.

1. get_service_price() – Retrieves the price of a business service.
2. check_service_available() – Checks whether a requested service is available.

I am using type hints and docstrings to make the functions
clear and reusable.

These functions will be used as tools by the AI agent in Day 4.

In [35]:
def get_service_price(service: str) -> int:
    """
    Return the price of a DemandPulse AI service.

    Args:
        service (str): Name of the requested service.

    Returns:
        int: Price of the service in rupees.

    Raises:
        ValueError: If the service is not available.
    """

    prices = {
        "sales forecasting": 500,
        "demand analysis": 300,
        "inventory planning": 400,
        "business report": 200
    }

    service_name = service.strip().lower()

    if service_name not in prices:
        raise ValueError("Service not found.")

    return prices[service_name]


def check_service_available(service: str) -> bool:
    """
    Check whether DemandPulse AI provides a particular service.

    Args:
        service (str): Name of the requested service.

    Returns:
        bool: True if the service is available, otherwise False.
    """

    available_services = [
        "sales forecasting",
        "demand analysis",
        "inventory planning",
        "business report"
    ]

    return service.strip().lower() in available_services

## Testing the Python Functions

In this step, I am testing both functions with valid and invalid
service names to verify that they work correctly.

In [36]:
print("Service Price:")
print("Sales Forecasting:", get_service_price("Sales Forecasting"))
print("Demand Analysis:", get_service_price("Demand Analysis"))

print("\nService Availability:")
print("Inventory Planning:", check_service_available("Inventory Planning"))
print("Financial Auditing:", check_service_available("Financial Auditing"))

Service Price:
Sales Forecasting: 500
Demand Analysis: 300

Service Availability:
Inventory Planning: True
Financial Auditing: False
